# Session 1 — SciPy: Distributions & Hypothesis Testing
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Every risk model starts with a distributional assumption. Every backtest result needs to be tested for statistical significance. Every factor needs its alpha tested against the null.

SciPy is the toolkit. But more importantly — knowing **which test to use and why** is what separates quant analysts from people who just run code.

> ⚠️ The biggest trap in quant finance: treating p < 0.05 as "the strategy works." In finance with small samples, non-IID returns, and multiple testing, p-values are dangerous if misread.


---
## 1️⃣ The Normal Distribution — And Why Finance Breaks It

The normal distribution assumes:
- Symmetric returns
- Thin tails (rare events are very rare)
- IID (independent, identically distributed) observations

**Financial returns violate all three:**

| Property | Normal Assumption | Reality |
|----------|------------------|---------|
| Tails | Thin | **Fat** — crashes happen far more often than predicted |
| Skew | 0 (symmetric) | **Negative** — big drops > big rises |
| IID | Independent | **Autocorrelated** — vol clusters, trends persist |
| Kurtosis | 3 | **>3** — excess kurtosis (leptokurtic) |

> 💡 **Why it still matters:** Most tools (OLS, Sharpe ratio, VaR) assume normality. Knowing when that assumption breaks tells you when your risk model is lying to you.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Download SPY and compute log returns
spy = yf.download('SPY', start='2010-01-01', auto_adjust=True)['Close'].squeeze()
log_ret = np.log(spy / spy.shift(1)).dropna()

print(f"Sample size: {len(log_ret)}")
print(f"Mean daily return:  {log_ret.mean():.6f}")
print(f"Std daily return:   {log_ret.std():.6f}")
print(f"Skewness:           {log_ret.skew():.4f}  (negative = left tail heavier)")
print(f"Excess Kurtosis:    {log_ret.kurtosis():.4f}  (>0 = fat tails vs normal)")
print(f"\nNormal dist would predict kurtosis = 0 (excess)")
print(f"SPY shows: {log_ret.kurtosis():.2f} — fat tails confirmed")


In [ ]:
# Fit a normal distribution to the returns
mu, sigma = stats.norm.fit(log_ret)

# Fit a Student-t distribution (handles fat tails better)
# 3 parameters: degrees of freedom (df), loc (mean), scale (std)
df_t, loc_t, scale_t = stats.t.fit(log_ret)

print(f"Normal fit:    mu={mu:.6f}, sigma={sigma:.6f}")
print(f"Student-t fit: df={df_t:.2f}, loc={loc_t:.6f}, scale={scale_t:.6f}")
print(f"\nDegrees of freedom={df_t:.1f}")
print("→ Low df = fat tails. df > 30 ≈ normal. Finance typically shows df = 3-6.")


In [ ]:
# VaR comparison: Normal vs Student-t
# VaR = loss not exceeded with probability (1-confidence)
confidence = 0.99
portfolio_value = 1_000_000

# Normal VaR
var_normal = stats.norm.ppf(1 - confidence, loc=mu, scale=sigma)
# Student-t VaR
var_t = stats.t.ppf(1 - confidence, df=df_t, loc=loc_t, scale=scale_t)
# Historical VaR (no distributional assumption)
var_hist = log_ret.quantile(1 - confidence)

print(f"99% 1-day VaR on ${portfolio_value:,.0f} portfolio:")
print(f"  Normal:      ${abs(var_normal) * portfolio_value:,.0f}  ({var_normal*100:.2f}%)")
print(f"  Student-t:   ${abs(var_t) * portfolio_value:,.0f}  ({var_t*100:.2f}%)")
print(f"  Historical:  ${abs(var_hist) * portfolio_value:,.0f}  ({var_hist*100:.2f}%)")
print()
print("→ Normal VaR UNDERSTATES risk vs Student-t and Historical.")
print("→ This is why Basel III requires stressed VaR, not just normal VaR.")


---
## 2️⃣ Hypothesis Testing — The Right Mental Model

**The workflow:**
1. State the null hypothesis H₀ (what you're trying to disprove)
2. Compute a test statistic
3. Find the p-value (probability of seeing this result if H₀ were true)
4. Reject H₀ if p < significance level (typically 0.05)

**In quant finance, common nulls:**
| Test | H₀ | When to use |
|------|-----|-------------|
| One-sample t-test | Strategy mean return = 0 | Is our alpha real? |
| Two-sample t-test | Two strategies have same mean | Does regime A ≠ regime B? |
| Shapiro-Wilk | Returns are normally distributed | Should I trust my VaR model? |
| Kolmogorov-Smirnov | Returns follow distribution X | Does a t-distribution fit better? |
| Ljung-Box | No autocorrelation in residuals | Are my model residuals clean? |

> ⚠️ **p-value ≠ probability the strategy works.** p < 0.05 means: IF the null were true, you'd see this result less than 5% of the time. It says nothing about effect size or economic significance.


In [ ]:
# Test 1: Is SPY's mean daily return significantly different from zero?
# H0: mean return = 0 (no alpha, just noise)
t_stat, p_value = stats.ttest_1samp(log_ret, popmean=0)

print("One-sample t-test: H0 = mean daily return is 0")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.6f}")
print(f"  Result: {'Reject H0 — return is significantly > 0' if p_value < 0.05 else 'Fail to reject H0'}")
print()

# But also compute the annualised effect size — statistical ≠ economic significance
annual_return = log_ret.mean() * 252
annual_vol    = log_ret.std() * np.sqrt(252)
sharpe        = annual_return / annual_vol
print(f"Annualised return: {annual_return*100:.2f}%")
print(f"Annualised vol:    {annual_vol*100:.2f}%")
print(f"Sharpe ratio:      {sharpe:.2f}")
print("→ Statistical significance + Sharpe > 0.5 = economically meaningful")


In [ ]:
# Test 2: Normality test (Shapiro-Wilk)
# H0: data is normally distributed
# Use a sample — Shapiro-Wilk loses power on very large samples
sample = log_ret.sample(500, random_state=42)
stat_sw, p_sw = stats.shapiro(sample)

print("Shapiro-Wilk normality test (n=500 sample)")
print(f"  Statistic: {stat_sw:.6f}")
print(f"  p-value:   {p_sw:.6f}")
print(f"  Result: {'Reject normality — fat tails confirmed' if p_sw < 0.05 else 'Cannot reject normality'}")
print()

# Kolmogorov-Smirnov: does our data fit a normal distribution?
ks_stat, ks_p = stats.kstest(log_ret, 'norm', args=(mu, sigma))
print("KS test: do returns fit Normal(mu, sigma)?")
print(f"  KS statistic: {ks_stat:.4f}")
print(f"  p-value:      {ks_p:.6f}")
print(f"  Result: {'Reject — returns do NOT follow normal dist' if ks_p < 0.05 else 'Cannot reject'}")


In [ ]:
# Test 3: Two-sample t-test — does crisis vol differ from calm vol?
# Split into pre-COVID and post-COVID periods
pre_covid  = log_ret['2015-01-01':'2020-02-01']
post_covid = log_ret['2020-03-01':'2023-12-31']

# Test if means differ
t2, p2 = stats.ttest_ind(pre_covid, post_covid)
print("Two-sample t-test: pre-COVID vs post-COVID mean returns")
print(f"  Pre-COVID mean:  {pre_covid.mean()*252*100:.2f}% annual")
print(f"  Post-COVID mean: {post_covid.mean()*252*100:.2f}% annual")
print(f"  p-value: {p2:.4f}")
print(f"  Result: {'Significant difference' if p2 < 0.05 else 'No significant difference'}")
print()

# Test if volatilities differ (Levene test — robust to non-normality)
lev_stat, lev_p = stats.levene(pre_covid, post_covid)
print("Levene test: pre-COVID vs post-COVID variance")
print(f"  Pre-COVID vol:  {pre_covid.std()*np.sqrt(252)*100:.1f}% annual")
print(f"  Post-COVID vol: {post_covid.std()*np.sqrt(252)*100:.1f}% annual")
print(f"  p-value: {lev_p:.4f}")
print(f"  Result: {'Significantly different volatility regimes' if lev_p < 0.05 else 'No significant vol difference'}")


---
## 3️⃣ Multiple Testing — The Silent Killer in Quant Research

If you test 20 uncorrelated strategies at p < 0.05, you expect **1 false positive by chance alone**.

In quant research where hundreds of parameter combinations are tested:
- Published Sharpe ratios are often inflated by selection bias
- The "best" backtest result is not the expected future result

**Corrections:**
| Method | When to use |
|--------|-------------|
| Bonferroni | Conservative — divide alpha by number of tests |
| Benjamini-Hochberg (FDR) | Better for many tests — controls false discovery rate |
| Deflated Sharpe Ratio | Correct Sharpe for number of trials (López de Prado) |


In [ ]:
# Multiple testing demo
np.random.seed(42)
n_tests = 100
n_obs = 252  # 1 year of daily returns

# Simulate 100 strategies with ZERO true alpha (pure noise)
p_values = []
for _ in range(n_tests):
    fake_returns = np.random.normal(0, 0.01, n_obs)
    _, p = stats.ttest_1samp(fake_returns, 0)
    p_values.append(p)

p_values = np.array(p_values)
naive_significant = (p_values < 0.05).sum()

print(f"100 strategies, all pure noise (zero true alpha)")
print(f"Naive p<0.05 'significant': {naive_significant} strategies")
print(f"Expected false positives:   {int(n_tests * 0.05)} (5% of 100)")
print()

# Bonferroni correction
bonferroni_alpha = 0.05 / n_tests
bonferroni_significant = (p_values < bonferroni_alpha).sum()
print(f"After Bonferroni correction (alpha={bonferroni_alpha:.4f}):")
print(f"  Significant: {bonferroni_significant}")
print()
print("→ Always apply multiple testing correction when scanning strategy parameters.")


---
## ✅ Self-Check Questions

1. Why does the Student-t distribution fit financial returns better than normal?
   > *Fat tails — crashes and spikes happen far more often than a normal distribution predicts. Student-t has a degrees-of-freedom parameter that controls tail weight.*

2. What does p < 0.05 actually mean?
   > *If the null hypothesis were true, you'd observe a result this extreme less than 5% of the time. It does NOT mean the strategy has a 95% chance of working.*

3. What's the difference between statistical significance and economic significance?
   > *A strategy can be statistically significantly positive but have a Sharpe of 0.1 — not worth trading. Always check effect size, not just p-values.*

4. Why is multiple testing dangerous in quant research?
   > *Testing 100 parameter combinations at p<0.05 produces ~5 false positives by chance. The "best" result from a parameter sweep is biased upward.*

---
## 🧪 Optional Tasks

- Compute skewness and kurtosis for 5 different assets (SPY, BTC, GLD, TLT, VIX futures). Which has the fattest tails?
- Run a Shapiro-Wilk test on weekly returns instead of daily. Is normality more plausible at lower frequency?
- Simulate the multiple testing problem with 50 strategies. Apply Bonferroni correction. How many survive?
- Compute 95% and 99% VaR for SPY using Normal, Student-t, and Historical methods. Plot the differences.
